# 05 - Preparar APRX para publicacion

Quinta etapa del flujo Geosupport. Agrega al APRX las imagenes cargadas como capas raster referenciadas fuera del Image Server, dentro del grupo `Vuelos Drone PAO > Imagenes Drone`, renombra las capas y ordena de la mas nueva a la mas vieja.

Por defecto guarda una copia de revision del APRX y no sobreescribe el original.

In [ ]:
from datetime import datetime
from pathlib import Path
import csv
import re
import sys

import pandas as pd
import arcpy

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'core').exists():
    for candidate in [Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'core').exists():
            PROJECT_ROOT = candidate
            break

if not (PROJECT_ROOT / 'core').exists():
    raise FileNotFoundError('No se encontro el folder core. Ejecuta el notebook desde la raiz del proyecto o desde flujo_geosupport_etapas.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FLOW_DIR = PROJECT_ROOT / 'flujo_geosupport_etapas'
print('Proyecto raiz:', PROJECT_ROOT)
print('Folder flujo:', FLOW_DIR)

## Parametros

In [ ]:
LOAD_RESULTS_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '02_load_results.csv'
LOAD_INPUT_ATTRIBUTES_CSV = FLOW_DIR / 'outputs' / 'etapa_02_carga_datastore_mosaico' / '01_load_input_with_attributes.csv'

APRX_PATH = r"\\amssclgis08.ams.gmams.cl\CL_MLP_PAO\01_Proyectos_ArcGIS\APRX\VISOR TERRITORIAL SIG PAO v7.aprx"
MAP_NAME = 'CL MLP PAO 27 Imagenes Aereas PAO Image Server'
PARENT_GROUP_NAME = 'Vuelos Drone PAO'
TARGET_GROUP_NAME = 'Imagenes Drone'

PREFIX = 'CL_MLP_PAO_IF_Ortho_'
SUFFIX = '.tif'

ADD_ONLY_MISSING_TO_GROUP = True
SAVE_COPY_FOR_REVIEW = True
SAVE_ORIGINAL_APRX = False
APRX_COPY_PATH = PROJECT_ROOT / 'APRX' / f"{Path(APRX_PATH).stem}_verificacion_imagenes_drone.aprx"

OUTPUT_DIR = FLOW_DIR / 'outputs' / 'etapa_05_preparar_aprx_publicacion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run_timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

print('CSV carga etapa 2:', LOAD_RESULTS_CSV)
print('APRX:', APRX_PATH)
print('Mapa:', MAP_NAME)
print('Grupo destino:', f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}')
print('Guardar copia revision:', SAVE_COPY_FOR_REVIEW)
print('Guardar APRX original:', SAVE_ORIGINAL_APRX)
print('Copia APRX:', APRX_COPY_PATH)

## 1. Leer imagenes cargadas

In [ ]:
def read_loaded_image_paths():
    if LOAD_RESULTS_CSV.exists():
        df = pd.read_csv(LOAD_RESULTS_CSV)
        if 'destination_path' not in df.columns and LOAD_INPUT_ATTRIBUTES_CSV.exists():
            attrs = pd.read_csv(LOAD_INPUT_ATTRIBUTES_CSV)
            df = df.merge(attrs[['Name', 'destination_path']], on='Name', how='left')
        status_mask = pd.Series(True, index=df.index)
        if 'overall_status' in df.columns:
            status_mask = df['overall_status'].astype(str).str.lower().isin(['ok'])
        if 'mosaic_add_status' in df.columns:
            status_mask = status_mask | df['mosaic_add_status'].astype(str).str.lower().isin(['added', 'already_exists'])
        df = df[status_mask & df['destination_path'].notna()].copy()
        return df['destination_path'].astype(str).drop_duplicates().tolist()

    if LOAD_INPUT_ATTRIBUTES_CSV.exists():
        df = pd.read_csv(LOAD_INPUT_ATTRIBUTES_CSV)
        path_field = 'destination_path' if 'destination_path' in df.columns else 'Path_Destino'
        return df[path_field].astype(str).dropna().drop_duplicates().tolist()

    raise FileNotFoundError('No existe salida de etapa 2 para actualizar APRX.')


image_paths_to_add = read_loaded_image_paths()
print(f'Imagenes para revisar/agregar en APRX: {len(image_paths_to_add)}')
for path in image_paths_to_add[:20]:
    print(path)

## 2. Funciones APRX

In [ ]:
def layer_long_name(layer):
    try:
        return layer.longName
    except Exception:
        return layer.name


def short_image_name(value):
    name = Path(str(value)).name
    while name.startswith('tmp_'):
        name = name.replace('tmp_', '', 1)
    if name.startswith(PREFIX):
        name = name.replace(PREFIX, '', 1)
    if name.endswith(SUFFIX):
        name = name[:-len(SUFFIX)]
    return name


def image_keys(value):
    short = short_image_name(value)
    return {
        short.lower(),
        f'{PREFIX}{short}'.lower(),
        f'{PREFIX}{short}{SUFFIX}'.lower(),
        f'{short}{SUFFIX}'.lower(),
    }


def image_date_sort_key(name):
    short = short_image_name(name)
    match = re.match(r'^(\d{2})_(\d{2})_(\d{2})_(.+)$', short)
    if not match:
        return (0, 0, 0, short.lower())
    yy, mm, dd, rest = match.groups()
    return (int(yy), int(mm), int(dd), rest.lower())


def find_group(map_obj, group_name, parent_group=None):
    groups = [layer for layer in map_obj.listLayers() if layer.isGroupLayer]
    matches = []
    for group in groups:
        if group.name != group_name:
            continue
        if parent_group is not None:
            expected_prefix = layer_long_name(parent_group) + '\\'
            if not layer_long_name(group).startswith(expected_prefix):
                continue
        matches.append(group)
    if not matches:
        available = [layer_long_name(layer) for layer in groups]
        parent_text = f' bajo {layer_long_name(parent_group)}' if parent_group else ''
        raise ValueError(f"No se encontro grupo '{group_name}'{parent_text}. Grupos disponibles: {available}")
    if len(matches) > 1:
        print(f"Advertencia: se encontraron {len(matches)} grupos '{group_name}'. Se usara {layer_long_name(matches[0])}")
    return matches[0]


def is_direct_child(layer, group_layer):
    long_name = layer_long_name(layer)
    group_long_name = layer_long_name(group_layer)
    prefix = group_long_name + '\\'
    if not long_name.startswith(prefix):
        return False
    relative = long_name[len(prefix):]
    return '\\' not in relative


def direct_raster_children(map_obj, group_layer):
    return [
        layer for layer in map_obj.listLayers()
        if layer != group_layer and is_direct_child(layer, group_layer) and layer.isRasterLayer and not layer.isGroupLayer
    ]


def add_raster_directly_to_group(map_obj, group_layer, raster_path, target_name):
    before = {layer_long_name(layer) for layer in direct_raster_children(map_obj, group_layer)}
    temp_layer = map_obj.addDataFromPath(raster_path)
    temp_layer.name = target_name
    try:
        map_obj.addLayerToGroup(group_layer, temp_layer, 'TOP')
    finally:
        map_obj.removeLayer(temp_layer)

    after_layers = direct_raster_children(map_obj, group_layer)
    new_layers = [layer for layer in after_layers if layer_long_name(layer) not in before]
    if new_layers:
        new_layers[0].name = target_name
        return new_layers[0]

    target_keys = image_keys(target_name)
    for layer in after_layers:
        if image_keys(layer.name).intersection(target_keys):
            layer.name = target_name
            return layer
    return None


def move_to_top_inside_group(map_obj, group_layer, layer_to_move):
    children = direct_raster_children(map_obj, group_layer)
    if not children or children[0] == layer_to_move:
        return
    map_obj.moveLayer(children[0], layer_to_move, 'BEFORE')


def order_group_newest_first(map_obj, group_layer):
    ordered = sorted(direct_raster_children(map_obj, group_layer), key=lambda layer: image_date_sort_key(layer.name), reverse=True)
    for layer in reversed(ordered):
        move_to_top_inside_group(map_obj, group_layer, layer)


def normalize_group_layer_names(map_obj, group_layer):
    renamed = 0
    for layer in direct_raster_children(map_obj, group_layer):
        new_name = short_image_name(layer.name)
        if layer.name != new_name:
            print(f'Renombrando: {layer.name} -> {new_name}')
            layer.name = new_name
            renamed += 1
    return renamed

## 3. Actualizar APRX

In [ ]:
aprx_results = []
aprx = arcpy.mp.ArcGISProject(APRX_PATH)

try:
    maps = aprx.listMaps(MAP_NAME)
    if not maps:
        raise ValueError(f'No se encontro el mapa: {MAP_NAME}')

    map_obj = maps[0]
    parent_group = find_group(map_obj, PARENT_GROUP_NAME)
    target_group = find_group(map_obj, TARGET_GROUP_NAME, parent_group)

    print(f'Grupo padre encontrado: {layer_long_name(parent_group)}')
    print(f'Grupo destino encontrado: {layer_long_name(target_group)}')

    existing_keys = set()
    for layer in direct_raster_children(map_obj, target_group):
        existing_keys.update(image_keys(layer.name))

    added_count = 0
    skipped_count = 0
    error_count = 0

    for raster_path in image_paths_to_add:
        target_name = short_image_name(raster_path)
        target_keys = image_keys(target_name)
        item = {'raster_path': raster_path, 'target_name': target_name, 'status': None, 'error': ''}

        if ADD_ONLY_MISSING_TO_GROUP and target_keys.intersection(existing_keys):
            item['status'] = 'skipped_existing'
            skipped_count += 1
            aprx_results.append(item)
            print(f'Ya existe, se omite: {target_name}')
            continue

        try:
            layer = add_raster_directly_to_group(map_obj, target_group, raster_path, target_name)
            if layer is None:
                item['status'] = 'added_not_confirmed'
                print(f'Advertencia: capa agregada no confirmada por ArcPy: {target_name}')
            else:
                item['status'] = 'added'
            existing_keys.update(target_keys)
            added_count += 1
            print(f'Agregada: {target_name}')
        except Exception as exc:
            item['status'] = 'error'
            item['error'] = str(exc)
            error_count += 1
            print(f'ERROR agregando {raster_path}: {exc}')
        aprx_results.append(item)

    renamed_count = normalize_group_layer_names(map_obj, target_group)
    order_group_newest_first(map_obj, target_group)

    final_layers = direct_raster_children(map_obj, target_group)
    print('Orden final, primeras 20 capas:')
    for layer in final_layers[:20]:
        print(f'- {layer.name}')

    print(f'Resumen APRX: agregadas={added_count}, omitidas={skipped_count}, renombradas={renamed_count}, errores={error_count}, total_grupo={len(final_layers)}')

    if SAVE_COPY_FOR_REVIEW:
        Path(APRX_COPY_PATH).parent.mkdir(parents=True, exist_ok=True)
        aprx.saveACopy(str(APRX_COPY_PATH))
        print(f'Copia de revision guardada: {APRX_COPY_PATH}')
    if SAVE_ORIGINAL_APRX:
        aprx.save()
        print('APRX original guardado')
finally:
    del aprx

aprx_results_df = pd.DataFrame(aprx_results)
display(aprx_results_df.head(30))

## 4. Exportar resultados

In [ ]:
summary_df = pd.DataFrame([
    {'metric': 'run_timestamp', 'value': run_timestamp},
    {'metric': 'load_results_csv', 'value': str(LOAD_RESULTS_CSV)},
    {'metric': 'aprx_path', 'value': APRX_PATH},
    {'metric': 'map_name', 'value': MAP_NAME},
    {'metric': 'target_group', 'value': f'{PARENT_GROUP_NAME} > {TARGET_GROUP_NAME}'},
    {'metric': 'image_paths_count', 'value': len(image_paths_to_add)},
    {'metric': 'save_copy_for_review', 'value': SAVE_COPY_FOR_REVIEW},
    {'metric': 'aprx_copy_path', 'value': str(APRX_COPY_PATH)},
])

for status, count in aprx_results_df['status'].value_counts(dropna=False).items() if not aprx_results_df.empty else []:
    summary_df.loc[len(summary_df)] = {'metric': f'status_{status}', 'value': int(count)}

summary_csv = OUTPUT_DIR / '00_summary.csv'
results_csv = OUTPUT_DIR / '01_aprx_update_results.csv'

summary_df.to_csv(summary_csv, index=False, encoding='utf-8-sig')
aprx_results_df.to_csv(results_csv, index=False, encoding='utf-8-sig')

display(summary_df)
print('Outputs exportados en:', OUTPUT_DIR)